In [2]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from datasets import load_dataset, Video
from huggingface_hub import login

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
# Initialize MediaPipe Pose Landmarker
base_options = python.BaseOptions(
    model_asset_path='pose_landmarker_heavy.task',
    delegate=python.BaseOptions.Delegate.GPU
)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5)
pose = vision.PoseLandmarker.create_from_options(options)

class PoseLandmark:
    NOSE = 0
    LEFT_EYE = 2
    RIGHT_EYE = 5
    LEFT_EAR = 7
    RIGHT_EAR = 8
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    LEFT_ELBOW = 13
    RIGHT_ELBOW = 14
    LEFT_WRIST = 15
    RIGHT_WRIST = 16

def calculate_angle(a, b, c):
    """Calculate the angle between three points."""
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

I0000 00:00:1783804708.018633  643976 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1783804708.122389  643976 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite delegate for Metal.


In [4]:
def process_video(video_path):
    """
    Process a video file and extract summary posture statistics.
    """
    cap = cv2.VideoCapture(video_path)
    
    # Store key metrics over frames
    shoulder_slopes_abs = []
    head_centering_scores = []
    hand_speeds = []
    hand_to_face_dists = []
    crossed_arms_dists = []
    
    # New metrics for leaning/slouching and shifts
    shoulder_widths = []
    nose_shoulder_dists = []
    core_positions = []
    core_speeds = []
    prev_left_wrist = None
    prev_right_wrist = None
    
    frame_count = 0


    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        if frame_count % 3 != 0:
            continue
        
        # Convert the BGR image to RGBA (Required for Mac/Metal GPU Delegate)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGBA)
        
        # Process the image and find poses
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGBA, data=image)
        results = pose.detect(mp_image)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks[0]
            
            # Get coordinates (normalized 0.0 to 1.0)
            nose = landmarks[PoseLandmark.NOSE]
            l_shoulder = landmarks[PoseLandmark.LEFT_SHOULDER]
            r_shoulder = landmarks[PoseLandmark.RIGHT_SHOULDER]
            l_wrist = landmarks[PoseLandmark.LEFT_WRIST]
            r_wrist = landmarks[PoseLandmark.RIGHT_WRIST]
            
            mid_shoulder_x = (l_shoulder.x + r_shoulder.x) / 2
            mid_shoulder_y = (l_shoulder.y + r_shoulder.y) / 2
            
            # Head Centering (Deviation of nose X from center of shoulders)
            head_centering_scores.append(abs(nose.x - mid_shoulder_x))
            
            # Calculate absolute shoulder slope (tilt)
            # dy / dx
            dx = r_shoulder.x - l_shoulder.x
            dy = r_shoulder.y - l_shoulder.y
            if dx != 0:
                slope = dy / dx
                shoulder_slopes_abs.append(abs(slope))
                
            # Leaning (shoulder width changes as you lean in/out)
            shoulder_widths.append(abs(r_shoulder.x - l_shoulder.x))
            
            # Slouching (vertical distance from nose to shoulder midpoint)
            nose_shoulder_dists.append(abs(nose.y - mid_shoulder_y))
            
            # Hand speed calculation
            current_l_wrist = (l_wrist.x, l_wrist.y)
            current_r_wrist = (r_wrist.x, r_wrist.y)
            if prev_left_wrist and prev_right_wrist:
                l_speed = np.sqrt((current_l_wrist[0] - prev_left_wrist[0])**2 + (current_l_wrist[1] - prev_left_wrist[1])**2)
                r_speed = np.sqrt((current_r_wrist[0] - prev_right_wrist[0])**2 + (current_r_wrist[1] - prev_right_wrist[1])**2)
                hand_speeds.append((l_speed + r_speed) / 2)
            prev_left_wrist = current_l_wrist
            prev_right_wrist = current_r_wrist
            
            # Hand to Face Distance (Face Touching)
            l_hand_face_dist = np.sqrt((l_wrist.x - nose.x)**2 + (l_wrist.y - nose.y)**2)
            r_hand_face_dist = np.sqrt((r_wrist.x - nose.x)**2 + (r_wrist.y - nose.y)**2)
            hand_to_face_dists.append(min(l_hand_face_dist, r_hand_face_dist))
            
            # Crossed Arms (Distance from wrists to opposite shoulders)
            l_wrist_r_shoulder_dist = np.sqrt((l_wrist.x - r_shoulder.x)**2 + (l_wrist.y - r_shoulder.y)**2)
            r_wrist_l_shoulder_dist = np.sqrt((r_wrist.x - l_shoulder.x)**2 + (r_wrist.y - l_shoulder.y)**2)
            crossed_arms_dists.append((l_wrist_r_shoulder_dist + r_wrist_l_shoulder_dist) / 2)
            
            # Posture Shifts (Core speed)
            current_core = (mid_shoulder_x, mid_shoulder_y)
            if core_positions:
                prev_core = core_positions[-1]
                # Euclidean distance between core position in current and previous frame
                speed = np.sqrt((current_core[0] - prev_core[0])**2 + (current_core[1] - prev_core[1])**2)
                core_speeds.append(speed)
            core_positions.append(current_core)
            
    cap.release()
    
    if len(core_positions) == 0:
        return None # No poses detected in the video
        
    # Calculate summary statistics
    metrics = {
        'head_centering_score_mean': np.mean(head_centering_scores) if head_centering_scores else 0,
        'absolute_shoulder_slope_mean': np.mean(shoulder_slopes_abs) if shoulder_slopes_abs else 0,
        'shoulder_slope_var': np.var(shoulder_slopes_abs) if shoulder_slopes_abs else 0,
        # Leaning and Slouching features
        'shoulder_width_mean': np.mean(shoulder_widths) if shoulder_widths else 0,
        'shoulder_width_var': np.var(shoulder_widths) if shoulder_widths else 0,
        'nose_shoulder_dist_mean': np.mean(nose_shoulder_dists) if nose_shoulder_dists else 0,
        'nose_shoulder_dist_var': np.var(nose_shoulder_dists) if nose_shoulder_dists else 0,
        # Hand & Arm Features
        'hand_speed_mean': np.mean(hand_speeds) if hand_speeds else 0,
        'hand_to_face_touches': sum(1 for d in hand_to_face_dists if d < 0.1) if hand_to_face_dists else 0,
        'crossed_arms_score': np.mean(crossed_arms_dists) if crossed_arms_dists else 0,
        # Posture Shift features
        'core_speed_mean': np.mean(core_speeds) if core_speeds else 0,
        'posture_shift_count': sum(1 for s in core_speeds if s > 0.02) if core_speeds else 0,
        # High level Behavioral scores
        'engagement_score': (np.mean(shoulder_widths) / (np.var(head_centering_scores) + 0.1)) if shoulder_widths and head_centering_scores else 0,
        'agitation_score': (sum(1 for s in core_speeds if s > 0.02) + sum(1 for d in hand_to_face_dists if d < 0.1)) if core_speeds and hand_to_face_dists else 0
    }
    
    return metrics

In [ ]:
# 1. Authenticate with HuggingFace
hf_token = ""
if not hf_token:
    print("Error: hf_token environment variable not set.")
    print('Please set it using: hf_token="your_hf_token"')

if hf_token:
    print("Logging into Hugging Face...")
    login(token=hf_token)

Logging into Hugging Face...


In [6]:

# 2. Load the Dataset
print("Loading dataset AI4A-lab/RecruitView...")
try:
    dataset = load_dataset("AI4A-lab/RecruitView", split="train") 
    # CRITICAL: We must cast the 'video' column to not decode automatically.
    # Otherwise, it returns a VideoDecoder object instead of the file path.
    dataset = dataset.cast_column('video', Video(decode=False))
    print("Done Loading Dataset")
except Exception as e:
    print(f"Failed to load dataset: {e}")

Loading dataset AI4A-lab/RecruitView...
Done Loading Dataset


In [7]:
all_features = []

# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

print(f"Total videos to process: {len(dataset)}")

# 3. Process each video
# We will just process the first 5 for a dry run, adjust this as needed!
for i, item in enumerate(dataset):
    vid_id = item.get('id', f"video_{i}") 
    
    video_data = item.get('video', None)
    if video_data and isinstance(video_data, dict) and 'path' in video_data:
        video_path = video_data['path']
    elif isinstance(video_data, str):
        video_path = video_data
    else:
         print(f"Skipping index {i}: Could not find video path.")
         continue
        
    print(f"Processing ID: {vid_id} | Path: {video_path}")
    metrics = process_video(video_path)
    
    if metrics:
        metrics['id'] = vid_id
        all_features.append(metrics)
    else:
        print(f"Warning: No posture detected for ID: {vid_id}")
        
    if i >= 4: 
        print("Stopping after 5 videos for dry-run testing. Change or remove the condition to process all.")
        break

Total videos to process: 2011
Processing ID: 0001 | Path: /Users/mohdshiblikhan/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140/videos/vid_0001.mp4


W0000 00:00:1783804721.691013  643985 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1783804721.691013  643978 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processing ID: 0002 | Path: /Users/mohdshiblikhan/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140/videos/vid_0002.mp4
Processing ID: 0003 | Path: /Users/mohdshiblikhan/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140/videos/vid_0003.mp4
Processing ID: 0004 | Path: /Users/mohdshiblikhan/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140/videos/vid_0004.mp4
Processing ID: 0005 | Path: /Users/mohdshiblikhan/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140/videos/vid_0005.mp4
Stopping after 5 videos for dry-run testing. Change or remove the condition to process all.


In [16]:
# 4. Save to CSV
if all_features:
    df = pd.DataFrame(all_features)
    
    # Reorder columns to put ID first
    cols = ['id'] + [c for c in df.columns if c != 'id']
    df = df[cols]
    
    output_csv = "data/posture_features.csv"
    df.to_csv(output_csv, index=False)
    print(f"\nSuccess! Features saved to {output_csv}")
else:
    print("No features extracted.")


Success! Features saved to data/posture_features.csv


In [10]:
idk = pd.read_csv("data/posture_features.csv")

In [11]:
idk

,id,head_centering_score_mean,absolute_shoulder_slope_mean,shoulder_slope_var,shoulder_width_mean,shoulder_width_var,nose_shoulder_dist_mean,nose_shoulder_dist_var,hand_speed_mean,hand_to_face_touches,crossed_arms_score,core_speed_mean,posture_shift_count,engagement_score,agitation_score
0,1,0.015226,0.067795,0.000911,0.617142,0.000326,0.319457,0.000757,0.051677,0,1.166635,0.007216,26,6.163163,26
1,2,0.026388,0.028222,0.000374,0.494716,0.000136,0.368540,0.000295,0.019394,0,0.945726,0.003491,0,4.927802,0
2,3,0.029648,0.020284,0.000274,0.654854,0.000133,0.294240,0.000359,0.099787,0,1.170374,0.006622,4,6.541636,4
3,4,0.029971,0.059990,0.002663,0.493375,0.000915,0.372601,0.000730,0.024589,2,0.925966,0.005781,36,4.920305,38
4,5,0.027023,0.041458,0.000712,0.452183,0.000052,0.327611,0.000254,0.016504,0,0.947141,0.004335,0,4.511034,0


In [13]:
idk.describe()

,id,head_centering_score_mean,absolute_shoulder_slope_mean,shoulder_slope_var,shoulder_width_mean,shoulder_width_var,nose_shoulder_dist_mean,nose_shoulder_dist_var,hand_speed_mean,hand_to_face_touches,crossed_arms_score,core_speed_mean,posture_shift_count,engagement_score,agitation_score
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,3.000000,0.025651,0.043550,0.000987,0.542454,0.000312,0.336490,0.000479,0.042390,0.400000,1.031168,0.005489,13.200000,5.412788,13.600000
std,1.581139,0.006036,0.020240,0.000971,0.088103,0.000351,0.033486,0.000244,0.034987,0.894427,0.125656,0.001555,16.709279,0.884346,17.401149
min,1.000000,0.015226,0.020284,0.000274,0.452183,0.000052,0.294240,0.000254,0.016504,0.000000,0.925966,0.003491,0.000000,4.511034,0.000000
25%,2.000000,0.026388,0.028222,0.000374,0.493375,0.000133,0.319457,0.000295,0.019394,0.000000,0.945726,0.004335,0.000000,4.920305,0.000000
50%,3.000000,0.027023,0.041458,0.000712,0.494716,0.000136,0.327611,0.000359,0.024589,0.000000,0.947141,0.005781,4.000000,4.927802,4.000000
75%,4.000000,0.029648,0.059990,0.000911,0.617142,0.000326,0.368540,0.000730,0.051677,0.000000,1.166635,0.006622,26.000000,6.163163,26.000000
max,5.000000,0.029971,0.067795,0.002663,0.654854,0.000915,0.372601,0.000757,0.099787,2.000000,1.170374,0.007216,36.000000,6.541636,38.000000


In [15]:
idk.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            5 non-null      int64  
 1   head_centering_score_mean     5 non-null      float64
 2   absolute_shoulder_slope_mean  5 non-null      float64
 3   shoulder_slope_var            5 non-null      float64
 4   shoulder_width_mean           5 non-null      float64
 5   shoulder_width_var            5 non-null      float64
 6   nose_shoulder_dist_mean       5 non-null      float64
 7   nose_shoulder_dist_var        5 non-null      float64
 8   hand_speed_mean               5 non-null      float64
 9   hand_to_face_touches          5 non-null      int64  
 10  crossed_arms_score            5 non-null      float64
 11  core_speed_mean               5 non-null      float64
 12  posture_shift_count           5 non-null      int64  
 13  engagement_score    

In [17]:
ik = pd.read_csv("data/posture_features.csv")

In [18]:
ik

,id,head_centering_score_mean,absolute_shoulder_slope_mean,shoulder_slope_var,shoulder_width_mean,shoulder_width_var,nose_shoulder_dist_mean,nose_shoulder_dist_var,hand_speed_mean,hand_to_face_touches,crossed_arms_score,core_speed_mean,posture_shift_count,engagement_score,agitation_score
0,1,0.014816,0.067882,0.000910,0.614849,0.000336,0.317518,0.000753,0.064670,0,1.170827,0.009070,12,6.139467,12
1,2,0.027182,0.027398,0.000319,0.492021,0.000151,0.368465,0.000285,0.026780,0,0.942724,0.004449,0,4.901325,0
2,3,0.028373,0.017856,0.000249,0.654566,0.000125,0.293514,0.000390,0.112922,0,1.168863,0.007432,2,6.539558,2
3,4,0.029751,0.060096,0.002636,0.493368,0.000793,0.371943,0.000747,0.039552,0,0.926297,0.010387,29,4.920241,29
4,5,0.026574,0.040499,0.000830,0.452547,0.000058,0.328449,0.000260,0.017858,0,0.943265,0.004824,0,4.514356,0
